# Inconsistent condition

The story is split into blocks (a line with `###` between them). Over the **whole story**:
- **consonants**: every t and every k is replaced by one of its alternatives, drawn at random (33 % each by default; one draw per word, so a word with two t's uses the same alternative for both)
- **vowels**: exactly the number of **words** and **phonemes** you choose for each vowel is changed, the words are drawn at random over the whole story, and a word never gets more than one vowel change

Each block can be spoken by a **different voice**, and the timing can be taken from a **reference voice reading the original text** (so every voice and every condition has the same timing).

The output is written to disk (an HTML player and a wav per block and for the whole condition); nothing is displayed in the notebook.

All functions are in `accent_tools.py` (next to `functions.py`); `help(make_inconsistent)`, `help(render_condition)` … show their documentation.

In [1]:
from accent_tools import *
from pathlib import Path
import torch
VOICES_DIR = Path("custom_voices")


c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


## Settings
Everything you need to change is in this cell.

In [2]:
# ---- story --------------------------------------------------------------------------------------------
STORY_FILE = "story2.txt"     # blocks separated by a line containing only ###
STORY_NAME = None             # name used in titles and file names (None = file name, e.g. "story1")
BLOCK_SEP  = "###"

# ---- voices -------------------------------------------------------------------------------------------
# BLOCK_VOICES = ["im_nicola", "if_sara", "im_nicola"]   # one voice per block (or one string for all blocks)
REF_VOICE    = "im_nicola"                             # reference voice for the timing (see below), or None

vb1 = torch.load(VOICES_DIR / "nicola_20.pt",  weights_only=True)
vb2 = torch.load(VOICES_DIR / "nicola_50.pt",    weights_only=True)
vb3 = torch.load(VOICES_DIR / "nicola_80.pt",      weights_only=True)
BLOCK_VOICES = [vb1, vb2, vb3]
# BLOCKS = [
#     dict(voice=vb1, ref_voice="im_nicola", errors={"t": "θ",  "k": "ɡ",  "i": "y"}),   # block 1
#     dict(voice=vb2,   ref_voice="im_nicola", errors={"t": "tʲ", "k": "x",  "e": "ø"}),   # block 2
#     dict(voice=vb3, ref_voice="im_nicola", errors={"t": "ɖ",  "k": "kʲ", "o": "ə"}),   # block 3
# ]


# ---- timing and prosody --------------------------------------------------------------------------
# The reference voice reads the ORIGINAL (unmanipulated) text; its timing is put on every block.
#   None             -> no transfer: every block voice times the accented text itself
#   "im_nicola"      -> the same reference voice for all blocks
#   [v1, v2, v3]     -> one reference voice per block
TRANSFER = dict(duration=True,   # take the reference voice's durations
                pitch=False,     # take its pitch contour too (moved into the block voice's range)
                energy=False)    # take its energy contour too
DUR_MODE = "raw"     # "raw":  exactly the reference voice's frames -> identical timing for every voice/condition
                     # "rate": the reference rhythm, stretched to each block voice's own speaking rate
# In a palatalized consonant (kʲ, tʲ ...) the consonant gets the reference duration and the added ʲ keeps
# the block voice's own duration (KEEP_OWN). Add "ʰ" to treat aspiration the same way, or () to split.
KEEP_OWN = ("ʲ",)

# ---- errors -------------------------------------------------------------------------------------------
# consonants: alternatives as a list (equal chance) or a dict {alternative: weight}
CONSONANTS = {"t": ["tʲ", "θ", "ɖ"],
              "k": ["x", "ɡ", "kʲ"]}               # ɡ = U+0261, the g in Kokoro's vocab (not the keyboard g)
SAME_IN_WORD = True                                # all t's (k's) of one word get the same alternative

# vowels: replacement, number of WORDS and number of PHONEMES to change over the whole story
VOWELS = {"e": dict(new="ø", words=248, phonemes=267),
          "i": dict(new="y", words=217, phonemes=236),
          "o": dict(new="ə", words=248, phonemes=304)}
OCCURRENCES = "all"      # "all":     every e of a chosen word changes; phonemes reached by choosing words with
                         #            more / fewer e's (exact unless impossible)
                         # "partial": exactly `phonemes` changes spread over the chosen words (some words keep
                         #            one of their e's)
SEED = 1                 # same seed -> same words and same alternatives

# ---- measurement (printed in the summary and saved in the _summary.csv) -----------------------------
# weight of each substitution for the weighted distance pw_w; pairs not listed = number of features
WEIGHTS = {("t", "tʲ"): 1, ("t", "θ"): 3, ("t", "ɖ"): 2,
           ("k", "x"):  3, ("k", "ɡ"): 2, ("k", "kʲ"): 1,
            ("e", "ø"):  1.5, ("i", "y"): 1.5, ("o", "ə"):  1.5}

# ---- output -------------------------------------------------------------------------------------------
OUT_DIR       = "output_online/inconsistent"
EXPORT_NATIVE = False                       # also export a native version with the same voices and timing
NATIVE_DIR    = "output_online/native"
N_PEEK        = 2                           # sentences per block shown in the IPA preview

## 1. Load the story
Splits the story into blocks at the `###` lines, converts every sentence to IPA and turns every ɪ into i.
The printout shows the number of sentences and words per block, check that it matches your blocks.

In [3]:
story = load_story(STORY_FILE, name=STORY_NAME, block_sep=BLOCK_SEP)

story2: 3 block(s), block 1: 35 sentences / 732 words, block 2: 34 sentences / 734 words, block 3: 37 sentences / 739 words   (IPA clean-up: 235 × ɪ→i)


## 2. Manipulate
Consonants first, then the vowel quotas. The printout shows, for every vowel, how many candidate words the story has and how many words / phonemes were changed (with a warning if a quota cannot be reached).
The summary table then shows, per block, what changed and how much (`pw` = per-word normalized distance, every change = 1; `pw_w` = with your `WEIGHTS`; both in %, see `help(measure_distance)`).

In [4]:
cond = make_inconsistent(story, consonants=CONSONANTS, vowels=VOWELS, same_in_word=SAME_IN_WORD,
                         occurrences=OCCURRENCES, seed=SEED)
print_summary(cond, weights=WEIGHTS);

inconsistent (story2, seed 1):
word_quota: 1744 candidate words, 430 of them can take more than one target
   e → ø: 862 candidate words (1001 phonemes) -> 248 of 248 words, 267 phonemes (target 267)
   i → y: 667 candidate words (745 phonemes) -> 217 of 217 words, 236 phonemes (target 236)
   o → ə: 721 candidate words (838 phonemes) -> 248 of 248 words, 304 phonemes (target 304)

inconsistent · story2
  block  sent  words  changed    pw %  pw_w %   changes
  1        35    732      404    18.3    32.3   e→ø ×76, i→y ×87, k→x ×51, k→kʲ ×40, k→ɡ ×39, o→ə ×88, t→ɖ ×75, t→θ ×66, t→tʲ ×65
  2        34    734      408    17.1    30.5   e→ø ×94, i→y ×68, k→ɡ ×43, k→x ×38, k→kʲ ×28, o→ə ×97, t→ɖ ×90, t→θ ×71, t→tʲ ×65
  3        37    739      421    19.1    32.7   e→ø ×97, i→y ×81, k→kʲ ×48, k→ɡ ×42, k→x ×37, o→ə ×119, t→ɖ ×83, t→θ ×76, t→tʲ ×75
  all     106   2205     1233    18.2    31.8   


### Look at the accented IPA (optional)
The first sentences of every block, native above accented. All sentences are also saved in `_ipa.txt` at export.

In [5]:
for blk in cond["blocks"]:
    print(f"── block {blk['index']}")
    for nat, man in list(zip(blk["ipa"], blk["manip"]))[:N_PEEK]:
        print("   native  :", nat)
        print("   accented:", man)

── block 1
   native  : lˈuka benedˈetːi inseɲˈava matemˈatika in ʊn liʧˈɛo di milˈano, e fˈino a kwaɾantˈanni non avˌeva maj lavorˈato in kampˈaɲɲa.
   accented: lˈuxa benedˈeɖːi inseɲˈava maɖemˈaɖixa in ʊn liʧˈɛə di milˈano, ø fˈyno a ɡwaɾanθˈanny nən avˌøva maj lavorˈaθo yn kʲampˈaɲɲa.
   native  : dˈue ˈanni prˈima ˈuno ʣˈio ʎi avˌeva laʃˈato ˌʊna pˈikːola vˈiɲɲa in vˈal ardˈɛzia, ˌʊna vˈalle strˈetːa viʧˈino al konfˈine kon la zvˈiʦːera.
   accented: dˈuø ˈanni prˈima ˈunə ʣˈyo ʎy avˌeva laʃˈatʲo ˌʊna pˈixːola vˈiɲɲa in vˈal ardˈɛzia, ˌʊna vˈalle sɖrˈeɖːa viʧˈino al ɡənfˈine kʲən la zvˈiʦːera.
── block 2
   native  : il prˈimo ʤˈorno di lavˈɔro ˈada e arɾivˈata ˌalle sˈɛtːe del matːˈino, kwˈando il sˈole non avˌeva ankˈora raʤːˈunto il fjˈanko dˌella montˈaɲɲa.
   accented: il prˈimo ʤˈorno dy lavˈɔro ˈada ø arɾivˈatʲa ˌalle sˈɛtʲːe døl maɖːˈinə, kʲwˈando il sˈəle nən avˌøva anxˈora raʤːˈunθo il fjˈanɡo dˌølla monθˈaɲɲa.
   native  : a portˈato kon sˈe alkˌʊni atːrˈeʦːi ke lˈuka n

## 3. Synthesize
One call per block. This is the slow step (a few seconds per sentence on CPU).

In [6]:
results = render_condition(cond, voices=BLOCK_VOICES, ref_voice=REF_VOICE, **TRANSFER,
                           dur_mode=DUR_MODE, keep_own=KEEP_OWN)

  inconsistent block 1: voice voice_block1, duration from im_nicola reading the original text


  inconsistent block 2: voice voice_block2, duration from im_nicola reading the original text
  inconsistent block 3: voice voice_block3, duration from im_nicola reading the original text


## 4. Export
Writes everything to `OUT_DIR`; nothing is displayed here. Open the `.html` files in a browser.

| file | content |
|---|---|
| `<condition>_<story>_block1.html` … | interactive player of one block (word boxes, token durations, altered phonemes highlighted, pitch / energy panel, legend of the changes in that block) |
| `<condition>_<story>_block1.wav` … | audio of one block |
| `<condition>_<story>_full.html` / `.wav` | the whole condition, blocks one after the other |
| `<condition>_<story>_ipa.txt` | native and accented IPA of every sentence |
| `<condition>_<story>_summary.csv` | per block: voice, reference voice, duration, words, changes, distances |

In [7]:
export_condition(cond, results, OUT_DIR, weights=WEIGHTS);

written to output_online\inconsistent/ :
    inconsistent_story2_block1.html
    inconsistent_story2_block1.wav
    inconsistent_story2_block2.html
    inconsistent_story2_block2.wav
    inconsistent_story2_block3.html
    inconsistent_story2_block3.wav
    inconsistent_story2_full.html
    inconsistent_story2_full.wav
    inconsistent_story2_ipa.txt
    inconsistent_story2_summary.csv


## 5. (Optional) Native version with the same voices and timing
The same story without any error, with the same block voices and the same reference timing, so it can be compared with the accented condition one to one.

In [8]:
if EXPORT_NATIVE:
    native = make_native(story)
    native_results = render_condition(native, voices=BLOCK_VOICES, ref_voice=REF_VOICE, **TRANSFER,
                                      dur_mode=DUR_MODE, keep_own=KEEP_OWN)
    export_condition(native, native_results, NATIVE_DIR);